# 🔌 Tu Primer Servidor MCP — Notebook guiado

**Taller práctico · SDK oficial Python v2 · Colab / Jupyter**. Este notebook es autocontenido y usa un **cliente MCP en memoria real**, sin LLM, sin API keys, sin tráfico a servicios externos.

Al final compara con la versión por `stdio` del repositorio.

## 1. Objetivos y tres primitivas

**Tool**: operación invocable. **Resource**: información identificada por URI. **Prompt**: plantilla de mensajes.

El servidor MCP no es un agente ni un modelo de IA.

In [ ]:
import sys
print("Python", sys.version)
assert sys.version_info >= (3, 11), "Usa Python 3.11 o superior"

## 2. Instalar el SDK oficial v2

**Nota:** la instalación necesita conexión a Internet una sola vez; el ejemplo del servidor y cliente no consumirá tokens ni claves.

In [ ]:
%pip install "mcp[cli]>=2.0,<3"

## 3. Modelo de inventario ficticio

Precios en céntimos y validaciones explícitas.

In [ ]:
INVENTARIO = {"bicicleta": {"stock": 3, "precio_centimos": 85000}, "casco": {"stock": 0, "precio_centimos": 12000}}

def cotizar(producto: str, cantidad: int) -> dict:
    if type(cantidad) is not int or cantidad < 1 or cantidad > 100:
        return {"ok": False, "codigo": "CANTIDAD_INVALIDA"}
    item = INVENTARIO.get(producto)
    if item is None:
        return {"ok": False, "codigo": "PRODUCTO_NO_ENCONTRADO"}
    if item["stock"] < cantidad:
        return {"ok": False, "codigo": "STOCK_INSUFICIENTE"}
    return {"ok": True, "total_centimos": item["precio_centimos"] * cantidad}

assert cotizar("bicicleta", 2)["total_centimos"] == 170000

## 4. Crear el servidor MCP

`MCPServer` es la clase de alto nivel del SDK oficial v2 (no `FastMCP` de v1).

In [ ]:
from mcp.server import MCPServer
import json
mcp = MCPServer("Tienda-Colab-Educativa")

@mcp.tool()
def cotizar_producto(producto: str, cantidad: int) -> dict:
    """Cotiza producto ficticio sin modificar stock ni procesar pagos."""
    return cotizar(producto, cantidad)

@mcp.resource("inventario://catalogo")
def catalogo() -> str:
    """Datos públicos ficticios."""
    return json.dumps(INVENTARIO, ensure_ascii=False)

@mcp.prompt()
def guion_cotizacion(producto: str) -> str:
    """Guion educativo para consultar un producto."""
    return f"Consulta el stock y precio de {producto}; no inventes información ni aceptes pagos."

## 5. Cliente MCP en proceso

El cliente habla con el servidor real sin abrir puertos ni iniciar un subprocess; es una capacidad de prueba del SDK.

In [ ]:
from mcp import Client
async with Client(mcp, raise_exceptions=True) as client:
    tools = await client.list_tools()
    print("Tools", [t.name for t in tools.tools])
    quote = await client.call_tool("cotizar_producto", {"producto": "bicicleta", "cantidad": 2})
    print("Error MCP", quote.is_error, "Resultado estructurado", quote.structured_content)
    resources = await client.list_resources()
    print("Resources", [str(r.uri) for r in resources.resources])
    doc = await client.read_resource("inventario://catalogo")
    print("Resource", doc.contents[0])
    prompts = await client.list_prompts()
    print("Prompts", [p.name for p in prompts.prompts])
    template = await client.get_prompt("guion_cotizacion", {"producto": "bicicleta"})
    print("Prompt", template.messages[0])

## 6. Prueba negativa y reflexión

Un dominio puede devolver `ok:false` sin que el protocolo marque `is_error:true`; analiza ambos niveles.

In [ ]:
async with Client(mcp, raise_exceptions=True) as client:
    failed = await client.call_tool("cotizar_producto", {"producto":"casco", "cantidad":1})
    print("is_error", failed.is_error, "contenido", failed.structured_content)

## 7. Próximo paso: stdio real

Para experimentar con un transporte verdadero y el Inspector, clona el repositorio y ejecuta:

```bash
python 02_probar_cliente_mcp.py
mcp dev servidor_mcp.py
```

El notebook utiliza **conexión en proceso**, no stdio ni HTTP. La demo web también es independiente del servidor.

## 8. Actividades

1. Agrega `consultar_stock` como Tool.
2. Crea un Resource con la política ficticia.
3. Comprueba el caso de producto desconocido.
4. Explica dónde añadirías autenticación si fueras a desplegar HTTP.

[Ver retos](../ejercicios/retos.md) · [Fuentes](../docs/REFERENCIAS.md)